# Sinh bảng và biểu đồ cho Section 3 — Kết quả mô hình phân lớp

Notebook này là nguồn tái lập duy nhất cho các bảng và biểu đồ trong Section 3.

Quy trình:

1. Đọc các CSV kết quả K-Fold và train–test trong outputs_LV.
2. Kiểm tra schema, khóa dữ liệu và miền giá trị chỉ số.
3. Tính mọi thống kê bằng code; các trường hợp đồng hạng F1-macro được ghi nhận bằng cột n_tied_f1.
4. Xuất bảng trung gian ở định dạng CSV.
5. Sinh biểu đồ từ dữ liệu đã kiểm tra.
6. Ki?m tra t?nh ??y ?? c?a to?n b? b?ng CSV v? bi?u ?? ?? xu?t.

Toàn bộ artifact được lưu dưới outputs_LV/sec_03_classification_results/.

In [1]:
from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "outputs_LV").is_dir() and (candidate / "CT552_TraMy_Nguyen").is_dir():
            return candidate
    raise FileNotFoundError("Không tìm thấy repository root chứa outputs_LV và CT552_TraMy_Nguyen.")

REPO_ROOT = find_repo_root(Path.cwd())
KFOLD_INPUT = REPO_ROOT / "outputs_LV" / "baseline_kfold_model_results"
SPLIT_INPUT = REPO_ROOT / "outputs_LV" / "baseline_split_model_results"
RETENTION_INPUT = REPO_ROOT / "outputs_LV" / "delete_rare_class_total" / "tables"

EXPORT_ROOT = REPO_ROOT / "outputs_LV" / "sec_03_classification_results"
TABLE_ROOT = EXPORT_ROOT / "tables"
CHART_ROOT = EXPORT_ROOT / "charts"

TABLE_KFOLD = TABLE_ROOT / "kfold"
TABLE_SPLIT = TABLE_ROOT / "train_test"
TABLE_COMPARE = TABLE_ROOT / "comparison"
CHART_KFOLD = CHART_ROOT / "kfold"
CHART_SPLIT = CHART_ROOT / "train_test"
CHART_COMPARE = CHART_ROOT / "comparison"

for directory in [
    TABLE_KFOLD, TABLE_SPLIT, TABLE_COMPARE,
    CHART_KFOLD, CHART_SPLIT, CHART_COMPARE,
]:
    directory.mkdir(parents=True, exist_ok=True)

KFOLD_K_VALUES = [2, 4, 5, 6]
SPLIT_K_VALUES = [4, 5]
METRICS = ["accuracy", "balanced_accuracy", "f1_macro", "precision", "recall"]
REQUIRED_COLUMNS = ["dataset", "fs_method", "model", *METRICS]

DATASET_ORDER = [
    "CuMiDa-Ovary-6008",
    "CuMiDa-Brain-15824",
    "CuMiDa-Breast-26304",
    "CuMiDa-Leukemia-28497",
    "CuMiDa-Colorectal-41657",
    "CuMiDa-Breast-45827",
    "CuMiDa-Brain-50161",
    "GEO-Breast-20711",
    "GEO-Mesothelioma-29354",
    "GEO-Kidney-36895",
    "GEO-Colon-68468",
    "GEO-MultiCancer-68606",
]
DATASET_DISPLAY = {
    "CuMiDa-Ovary-6008": "GSE6008",
    "CuMiDa-Brain-15824": "GSE15824",
    "CuMiDa-Breast-26304": "GSE26304",
    "CuMiDa-Leukemia-28497": "GSE28497",
    "CuMiDa-Colorectal-41657": "GSE41657",
    "CuMiDa-Breast-45827": "GSE45827",
    "CuMiDa-Brain-50161": "GSE50161",
    "GEO-Breast-20711": "GSE20711",
    "GEO-Mesothelioma-29354": "GSE29354",
    "GEO-Kidney-36895": "GSE36895",
    "GEO-Colon-68468": "GSE68468",
    "GEO-MultiCancer-68606": "GSE68606",
}
FS_ORDER = ["raw", "mrmr_k50", "mrmr_k75", "boruta"]
FS_DISPLAY = {
    "raw": "Raw",
    "mrmr_k50": "mRMR-50",
    "mrmr_k75": "mRMR-75",
    "boruta": "Boruta",
}
MODEL_ORDER = ["nb", "knn", "svm", "dt", "rf", "xgboost", "ann"]
MODEL_DISPLAY = {
    "nb": "GNB",
    "knn": "KNN",
    "svm": "SVM",
    "dt": "DT",
    "rf": "RF",
    "xgboost": "XGBoost",
    "ann": "ANN",
}

plt.rcParams.update({
    "font.family": "DejaVu Serif",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "figure.dpi": 120,
    "savefig.dpi": 300,
})

print("REPO_ROOT:", REPO_ROOT)
print("EXPORT_ROOT:", EXPORT_ROOT)

REPO_ROOT: D:\LV_KHMT_code\Transcriptomic_XAI_thesis
EXPORT_ROOT: D:\LV_KHMT_code\Transcriptomic_XAI_thesis\outputs_LV\sec_03_classification_results


In [2]:
def load_sweep(input_dir: Path, k_values: list[int], flow_name: str) -> pd.DataFrame:
    frames = []
    for k in k_values:
        path = input_dir / f"k{k}.csv"
        if not path.exists():
            raise FileNotFoundError(f"Thiếu tệp đầu vào: {path}")
        frame = pd.read_csv(path)
        missing = [column for column in REQUIRED_COLUMNS if column not in frame.columns]
        if missing:
            raise ValueError(f"{path.name} thiếu các cột: {missing}")
        frame = frame[REQUIRED_COLUMNS].copy()
        frame["k"] = int(k)
        frame["flow"] = flow_name
        frames.append(frame)

    result = pd.concat(frames, ignore_index=True)
    key = ["dataset", "fs_method", "model", "k"]
    if result.duplicated(key).any():
        duplicated = result.loc[result.duplicated(key, keep=False), key]
        raise ValueError(f"Phát hiện khóa trùng trong {flow_name}:\n{duplicated}")

    for metric in METRICS:
        if result[metric].isna().any():
            raise ValueError(f"{flow_name}: cột {metric} có giá trị thiếu.")
        invalid = ~result[metric].between(0, 1)
        if invalid.any():
            raise ValueError(f"{flow_name}: cột {metric} có giá trị ngoài [0, 1].")

    unknown_datasets = sorted(set(result["dataset"]) - set(DATASET_ORDER))
    if unknown_datasets:
        raise ValueError(f"Bộ dữ liệu chưa có ánh xạ tên: {unknown_datasets}")

    return result

kfold = load_sweep(KFOLD_INPUT, KFOLD_K_VALUES, "kfold")
split = load_sweep(SPLIT_INPUT, SPLIT_K_VALUES, "train_test")

if sorted(kfold["dataset"].unique()) != sorted(DATASET_ORDER):
    raise ValueError("Danh sách bộ dữ liệu K-Fold không khớp DATASET_ORDER.")
if sorted(split["dataset"].unique()) != sorted(DATASET_ORDER):
    raise ValueError("Danh sách bộ dữ liệu train–test không khớp DATASET_ORDER.")

sample_retention = pd.read_csv(RETENTION_INPUT / "sample_count_remaining_by_k.csv")
class_retention = pd.read_csv(RETENTION_INPUT / "rare_class_summary_k0_2_3_4_5_6.csv")

print("K-Fold:", kfold.shape, "rows")
print("Train–test:", split.shape, "rows")
display(kfold.head())
display(split.head())

K-Fold: (1328, 10) rows
Train–test: (501, 10) rows


,dataset,fs_method,model,accuracy,balanced_accuracy,f1_macro,precision,recall,k,flow
0,CuMiDa-Brain-15824,boruta,ann,1.000000,1.000000,1.000000,1.000000,1.000000,2,kfold
1,CuMiDa-Brain-15824,boruta,dt,0.725000,0.716667,0.640000,0.616667,0.716667,2,kfold
2,CuMiDa-Brain-15824,boruta,knn,0.975000,0.983333,0.973333,0.975000,0.983333,2,kfold
3,CuMiDa-Brain-15824,boruta,nb,0.917857,0.900000,0.889524,0.904167,0.900000,2,kfold
4,CuMiDa-Brain-15824,boruta,rf,0.975000,0.983333,0.973333,0.975000,0.983333,2,kfold


,dataset,fs_method,model,accuracy,balanced_accuracy,f1_macro,precision,recall,k,flow
0,GEO-Breast-20711,boruta,nb,0.692308,0.7500,0.668452,0.7500,0.7500,4,train_test
1,GEO-Breast-20711,boruta,knn,0.692308,0.7500,0.678571,0.6875,0.7500,4,train_test
2,GEO-Breast-20711,boruta,svm,0.692308,0.6875,0.671429,0.6875,0.6875,4,train_test
3,GEO-Breast-20711,boruta,rf,0.692308,0.7500,0.678571,0.6875,0.7500,4,train_test
4,GEO-Breast-20711,boruta,dt,0.538462,0.5625,0.500000,0.5250,0.5625,4,train_test


In [3]:
FS_RANK = {name: index for index, name in enumerate(FS_ORDER)}
MODEL_RANK = {name: index for index, name in enumerate(MODEL_ORDER)}
DATASET_RANK = {name: index for index, name in enumerate(DATASET_ORDER)}

def representative_best(data: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    representatives = []
    all_ties = []

    for (dataset, k), group in data.groupby(["dataset", "k"], sort=False):
        maximum = float(group["f1_macro"].max())
        tied = group[np.isclose(group["f1_macro"], maximum, rtol=0, atol=1e-12)].copy()
        tied["_fs_rank"] = tied["fs_method"].map(FS_RANK)
        tied["_model_rank"] = tied["model"].map(MODEL_RANK)
        tied = tied.sort_values(
            ["balanced_accuracy", "accuracy", "_fs_rank", "_model_rank"],
            ascending=[False, False, True, True],
        )
        representative = tied.iloc[0].drop(labels=["_fs_rank", "_model_rank"]).to_dict()
        representative["n_tied_f1"] = int(len(tied))
        representatives.append(representative)

        tie_rows = tied.drop(columns=["_fs_rank", "_model_rank"]).copy()
        tie_rows["n_tied_f1"] = int(len(tied))
        all_ties.append(tie_rows)

    reps = pd.DataFrame(representatives)
    ties = pd.concat(all_ties, ignore_index=True)
    reps["_dataset_rank"] = reps["dataset"].map(DATASET_RANK)
    reps = reps.sort_values(["_dataset_rank", "k"]).drop(columns="_dataset_rank").reset_index(drop=True)
    return reps, ties

def dimension_summary(data: pd.DataFrame, dimension: str) -> pd.DataFrame:
    if dimension not in {"model", "fs_method"}:
        raise ValueError("dimension phải là model hoặc fs_method.")

    summary = (
        data.groupby(dimension)
        .agg(
            mean_f1=("f1_macro", "mean"),
            median_f1=("f1_macro", "median"),
            std_f1=("f1_macro", "std"),
            mean_balanced_accuracy=("balanced_accuracy", "mean"),
            n_results=("f1_macro", "size"),
            n_datasets=("dataset", "nunique"),
        )
        .reset_index()
    )

    _, ties = representative_best(data)
    best_counts = (
        ties[["dataset", "k", dimension]]
        .drop_duplicates()
        .groupby(dimension)
        .size()
        .rename("best_case_count_tie_inclusive")
        .reset_index()
    )
    summary = summary.merge(best_counts, on=dimension, how="left")
    summary["best_case_count_tie_inclusive"] = (
        summary["best_case_count_tie_inclusive"].fillna(0).astype(int)
    )

    order = MODEL_ORDER if dimension == "model" else FS_ORDER
    rank = {name: index for index, name in enumerate(order)}
    summary["_rank"] = summary[dimension].map(rank)
    return summary.sort_values("_rank").drop(columns="_rank").reset_index(drop=True)

kfold_best, kfold_ties = representative_best(kfold)
split_best_long, split_ties = representative_best(split)

def add_display_columns(frame: pd.DataFrame) -> pd.DataFrame:
    output = frame.copy()
    output["dataset_display"] = output["dataset"].map(DATASET_DISPLAY)
    output["fs_display"] = output["fs_method"].map(FS_DISPLAY)
    output["model_display"] = output["model"].map(MODEL_DISPLAY)
    output["representative_config"] = output["fs_display"] + "/" + output["model_display"]
    return output

kfold_best = add_display_columns(kfold_best)
split_best_long = add_display_columns(split_best_long)

split_k4 = split_best_long[split_best_long["k"] == 4].copy()
split_k5 = split_best_long[split_best_long["k"] == 5].copy()
split_best_wide = split_k4[
    ["dataset", "dataset_display", "representative_config", "f1_macro",
     "balanced_accuracy", "accuracy", "n_tied_f1"]
].merge(
    split_k5[
        ["dataset", "representative_config", "f1_macro",
         "balanced_accuracy", "accuracy", "n_tied_f1"]
    ],
    on="dataset",
    suffixes=("_k4", "_k5"),
)
split_best_wide["delta_f1_k5_minus_k4"] = (
    split_best_wide["f1_macro_k5"] - split_best_wide["f1_macro_k4"]
)
split_best_wide["_rank"] = split_best_wide["dataset"].map(DATASET_RANK)
split_best_wide = split_best_wide.sort_values("_rank").drop(columns="_rank").reset_index(drop=True)

sample_original_column = sample_retention.columns[1]
original_class_total = int(
    class_retention.loc[
        class_retention["rare_class_threshold_k"] == 0, "n_classes_remaining"
    ].sum()
)
threshold_rows = []
for k in KFOLD_K_VALUES:
    best_at_k = kfold_best[kfold_best["k"] == k]
    remaining_samples = int(sample_retention[f"k={k}"].sum())
    remaining_classes = int(
        class_retention.loc[
            class_retention["rare_class_threshold_k"] == k, "n_classes_remaining"
        ].sum()
    )
    threshold_rows.append({
        "k": k,
        "remaining_samples": remaining_samples,
        "remaining_classes": remaining_classes,
        "removed_samples_vs_original": int(sample_retention[sample_original_column].sum()) - remaining_samples,
        "removed_classes_vs_original": original_class_total - remaining_classes,
        "mean_best_f1": best_at_k["f1_macro"].mean(),
        "median_best_f1": best_at_k["f1_macro"].median(),
        "mean_best_balanced_accuracy": best_at_k["balanced_accuracy"].mean(),
    })
threshold_tradeoff = pd.DataFrame(threshold_rows)

kfold_model_all = dimension_summary(kfold, "model")
kfold_model_k4 = dimension_summary(kfold[kfold["k"] == 4], "model")
split_model_all = dimension_summary(split, "model")
split_model_k4 = dimension_summary(split[split["k"] == 4], "model")

kfold_fs_all = dimension_summary(kfold, "fs_method")
kfold_fs_k4 = dimension_summary(kfold[kfold["k"] == 4], "fs_method")
split_fs_all = dimension_summary(split, "fs_method")
split_fs_k4 = dimension_summary(split[split["k"] == 4], "fs_method")

fs_comparison_k4 = kfold_fs_k4.merge(
    split_fs_k4,
    on="fs_method",
    how="outer",
    suffixes=("_kfold", "_split"),
)
fs_comparison_k4["_rank"] = fs_comparison_k4["fs_method"].map(FS_RANK)
fs_comparison_k4 = (
    fs_comparison_k4.sort_values("_rank").drop(columns="_rank").reset_index(drop=True)
)

display(threshold_tradeoff)
display(kfold_best.head(8))
display(split_best_wide)
display(fs_comparison_k4)

,k,remaining_samples,remaining_classes,removed_samples_vs_original,removed_classes_vs_original,mean_best_f1,median_best_f1,mean_best_balanced_accuracy
0,2,1586,68,9,9,0.920499,0.937126,0.926289
1,4,1570,60,25,17,0.930008,0.956363,0.934577
2,5,1558,57,37,20,0.929868,0.955522,0.934661
3,6,1533,52,62,25,0.934806,0.973746,0.938365


,dataset,fs_method,model,accuracy,balanced_accuracy,f1_macro,precision,recall,k,flow,n_tied_f1,dataset_display,fs_display,model_display,representative_config
0,CuMiDa-Ovary-6008,mrmr_k50,svm,0.867895,0.902083,0.90395,0.932475,0.902083,2,kfold,1,GSE6008,mRMR-50,SVM,mRMR-50/SVM
1,CuMiDa-Ovary-6008,mrmr_k50,svm,0.867895,0.902083,0.90395,0.932475,0.902083,4,kfold,1,GSE6008,mRMR-50,SVM,mRMR-50/SVM
2,CuMiDa-Ovary-6008,mrmr_k50,svm,0.867895,0.902083,0.90395,0.932475,0.902083,5,kfold,1,GSE6008,mRMR-50,SVM,mRMR-50/SVM
3,CuMiDa-Ovary-6008,mrmr_k50,svm,0.867895,0.902083,0.90395,0.932475,0.902083,6,kfold,1,GSE6008,mRMR-50,SVM,mRMR-50/SVM
4,CuMiDa-Brain-15824,boruta,ann,1.000000,1.000000,1.00000,1.000000,1.000000,2,kfold,1,GSE15824,Boruta,ANN,Boruta/ANN
5,CuMiDa-Brain-15824,boruta,ann,1.000000,1.000000,1.00000,1.000000,1.000000,4,kfold,1,GSE15824,Boruta,ANN,Boruta/ANN
6,CuMiDa-Brain-15824,boruta,ann,1.000000,1.000000,1.00000,1.000000,1.000000,5,kfold,1,GSE15824,Boruta,ANN,Boruta/ANN
7,CuMiDa-Brain-15824,boruta,ann,1.000000,1.000000,1.00000,1.000000,1.000000,6,kfold,1,GSE15824,Boruta,ANN,Boruta/ANN


,dataset,dataset_display,representative_config_k4,f1_macro_k4,balanced_accuracy_k4,accuracy_k4,n_tied_f1_k4,representative_config_k5,f1_macro_k5,balanced_accuracy_k5,accuracy_k5,n_tied_f1_k5,delta_f1_k5_minus_k4
0,CuMiDa-Ovary-6008,GSE6008,mRMR-50/RF,0.866667,0.875000,0.800000,4,mRMR-50/RF,0.866667,0.875000,0.800000,4,0.000000
1,CuMiDa-Brain-15824,GSE15824,mRMR-50/RF,1.000000,1.000000,1.000000,4,mRMR-50/RF,1.000000,1.000000,1.000000,4,0.000000
2,CuMiDa-Breast-26304,GSE26304,Boruta/ANN,0.758974,0.775000,0.705882,1,Boruta/ANN,0.758974,0.775000,0.705882,1,0.000000
3,CuMiDa-Leukemia-28497,GSE28497,Boruta/ANN,0.939394,0.939394,0.953488,1,Boruta/ANN,0.939394,0.939394,0.953488,1,0.000000
4,CuMiDa-Colorectal-41657,GSE41657,mRMR-50/KNN,1.000000,1.000000,1.000000,1,mRMR-50/KNN,1.000000,1.000000,1.000000,1,0.000000
5,CuMiDa-Breast-45827,GSE45827,mRMR-50/GNB,1.000000,1.000000,1.000000,14,mRMR-50/GNB,1.000000,1.000000,1.000000,14,0.000000
6,CuMiDa-Brain-50161,GSE50161,mRMR-50/ANN,1.000000,1.000000,1.000000,10,mRMR-50/ANN,1.000000,1.000000,1.000000,10,0.000000
7,GEO-Breast-20711,GSE20711,Boruta/ANN,0.773810,0.812500,0.769231,1,Boruta/ANN,0.773810,0.812500,0.769231,1,0.000000
8,GEO-Mesothelioma-29354,GSE29354,mRMR-50/GNB,0.777778,0.777778,0.777778,3,mRMR-50/GNB,0.777778,0.777778,0.777778,3,0.000000
9,GEO-Kidney-36895,GSE36895,mRMR-50/DT,1.000000,1.000000,1.000000,1,Boruta/KNN,0.644444,0.666667,0.888889,3,-0.355556


,fs_method,mean_f1_kfold,median_f1_kfold,std_f1_kfold,mean_balanced_accuracy_kfold,n_results_kfold,n_datasets_kfold,best_case_count_tie_inclusive_kfold,mean_f1_split,median_f1_split,std_f1_split,mean_balanced_accuracy_split,n_results_split,n_datasets_split,best_case_count_tie_inclusive_split
0,raw,0.693283,0.720347,0.192203,0.709524,83,12,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,mrmr_k50,0.811074,0.841044,0.122950,0.826025,83,12,2,0.734736,0.777778,0.225467,0.758453,84.0,12.0,9.0
2,mrmr_k75,0.817521,0.858301,0.130133,0.832166,83,12,5,0.731467,0.777778,0.214549,0.756938,84.0,12.0,7.0
3,boruta,0.847484,0.876751,0.127934,0.860560,83,12,7,0.741077,0.773729,0.207842,0.768359,84.0,12.0,6.0


In [4]:
csv_exports = {
    TABLE_KFOLD / "best_by_dataset_k.csv": kfold_best[
        [
            "dataset", "dataset_display", "k", "fs_method", "fs_display",
            "model", "model_display", "representative_config", "n_tied_f1",
            "accuracy", "balanced_accuracy", "f1_macro", "precision", "recall",
        ]
    ],
    TABLE_KFOLD / "all_tied_best_configurations.csv": add_display_columns(kfold_ties),
    TABLE_KFOLD / "model_summary_all_k.csv": kfold_model_all,
    TABLE_KFOLD / "model_summary_k4.csv": kfold_model_k4,
    TABLE_KFOLD / "feature_selection_summary_all_k.csv": kfold_fs_all,
    TABLE_KFOLD / "feature_selection_summary_k4.csv": kfold_fs_k4,

    TABLE_SPLIT / "best_by_dataset_k.csv": split_best_long[
        [
            "dataset", "dataset_display", "k", "fs_method", "fs_display",
            "model", "model_display", "representative_config", "n_tied_f1",
            "accuracy", "balanced_accuracy", "f1_macro", "precision", "recall",
        ]
    ],
    TABLE_SPLIT / "best_k4_k5_wide.csv": split_best_wide,
    TABLE_SPLIT / "all_tied_best_configurations.csv": add_display_columns(split_ties),
    TABLE_SPLIT / "model_summary_all_k.csv": split_model_all,
    TABLE_SPLIT / "model_summary_k4.csv": split_model_k4,
    TABLE_SPLIT / "feature_selection_summary_all_k.csv": split_fs_all,
    TABLE_SPLIT / "feature_selection_summary_k4.csv": split_fs_k4,

    TABLE_COMPARE / "threshold_tradeoff.csv": threshold_tradeoff,
    TABLE_COMPARE / "feature_selection_k4.csv": fs_comparison_k4,
}

for path, frame in csv_exports.items():
    frame.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"CSV: {path.relative_to(REPO_ROOT)} ({len(frame)} rows)")

CSV: outputs_LV\sec_03_classification_results\tables\kfold\best_by_dataset_k.csv (48 rows)
CSV: outputs_LV\sec_03_classification_results\tables\kfold\all_tied_best_configurations.csv (90 rows)
CSV: outputs_LV\sec_03_classification_results\tables\kfold\model_summary_all_k.csv (7 rows)
CSV: outputs_LV\sec_03_classification_results\tables\kfold\model_summary_k4.csv (7 rows)
CSV: outputs_LV\sec_03_classification_results\tables\kfold\feature_selection_summary_all_k.csv (4 rows)
CSV: outputs_LV\sec_03_classification_results\tables\kfold\feature_selection_summary_k4.csv (4 rows)
CSV: outputs_LV\sec_03_classification_results\tables\train_test\best_by_dataset_k.csv (24 rows)
CSV: outputs_LV\sec_03_classification_results\tables\train_test\best_k4_k5_wide.csv (12 rows)
CSV: outputs_LV\sec_03_classification_results\tables\train_test\all_tied_best_configurations.csv (110 rows)
CSV: outputs_LV\sec_03_classification_results\tables\train_test\model_summary_all_k.csv (7 rows)
CSV: outputs_LV\sec_03_cla

In [5]:
def save_figure(fig: plt.Figure, path: Path) -> None:
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print("CHART:", path.relative_to(REPO_ROOT))

# 1) Heatmap Dataset × k của best F1-macro
heatmap_data = (
    kfold_best.pivot(index="dataset", columns="k", values="f1_macro")
    .reindex(index=DATASET_ORDER, columns=KFOLD_K_VALUES)
)
fig, ax = plt.subplots(figsize=(7.2, 8.0))
image = ax.imshow(heatmap_data.values, cmap="YlGnBu", vmin=0.60, vmax=1.00, aspect="auto")
ax.set_xticks(range(len(KFOLD_K_VALUES)), [f"k={k}" for k in KFOLD_K_VALUES])
ax.set_yticks(range(len(DATASET_ORDER)), [DATASET_DISPLAY[d] for d in DATASET_ORDER])
ax.set_xlabel("Ngưỡng loại lớp k")
ax.set_ylabel("Bộ dữ liệu")
ax.set_title("F1-macro tốt nhất theo bộ dữ liệu và ngưỡng k")
for row in range(heatmap_data.shape[0]):
    for column in range(heatmap_data.shape[1]):
        value = heatmap_data.iloc[row, column]
        color = "white" if value >= 0.88 else "black"
        ax.text(column, row, f"{value:.3f}", ha="center", va="center", color=color, fontsize=8.5)
colorbar = fig.colorbar(image, ax=ax, fraction=0.035, pad=0.03)
colorbar.set_label("F1-macro")
fig.tight_layout()
save_figure(fig, CHART_KFOLD / "best_f1_heatmap.png")

# 2) Grouped bar chart best F1 tại k=4 và k=5
x = np.arange(len(split_best_wide))
width = 0.38
fig, ax = plt.subplots(figsize=(12.0, 5.8))
bars_k4 = ax.bar(x - width / 2, split_best_wide["f1_macro_k4"], width, label="k=4", color="#4472C4")
bars_k5 = ax.bar(x + width / 2, split_best_wide["f1_macro_k5"], width, label="k=5", color="#ED7D31")
ax.set_xticks(x, split_best_wide["dataset_display"], rotation=45, ha="right")
ax.set_ylim(0.55, 1.04)
ax.set_ylabel("F1-macro tốt nhất")
ax.set_xlabel("Bộ dữ liệu")
ax.set_title("So sánh kết quả train–test giữa k=4 và k=5")
ax.legend()
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
save_figure(fig, CHART_SPLIT / "best_f1_k4_k5_grouped_bar.png")

# 3) Scatter K-Fold vs train–test tại k=4 và k=5
fig, axes = plt.subplots(1, 2, figsize=(11.5, 5.2), sharex=True, sharey=True)
for ax, k in zip(axes, SPLIT_K_VALUES):
    x_df = kfold_best[kfold_best["k"] == k][["dataset", "f1_macro"]].rename(
        columns={"f1_macro": "kfold_f1"}
    )
    y_df = split_best_long[split_best_long["k"] == k][["dataset", "f1_macro"]].rename(
        columns={"f1_macro": "split_f1"}
    )
    comparison = x_df.merge(y_df, on="dataset")
    ax.scatter(comparison["kfold_f1"], comparison["split_f1"], s=48, color="#2F5597", alpha=0.85)
    ax.plot([0.60, 1.01], [0.60, 1.01], linestyle="--", color="gray", linewidth=1)
    for row in comparison.itertuples(index=False):
        # Chỉ gắn nhãn các điểm lệch đáng kể khỏi đường y=x để tránh chồng chữ.
        if abs(row.split_f1 - row.kfold_f1) >= 0.025:
            ax.annotate(
                DATASET_DISPLAY[row.dataset],
                (row.kfold_f1, row.split_f1),
                xytext=(3, 3),
                textcoords="offset points",
                fontsize=6.8,
            )
    ax.set_title(f"k={k}")
    ax.set_xlabel("Best F1-macro — K-Fold")
    ax.grid(alpha=0.20)
axes[0].set_ylabel("Best F1-macro — train–test")
axes[0].set_xlim(0.60, 1.02)
axes[0].set_ylim(0.60, 1.02)
fig.suptitle("Đối chiếu mô tả giữa Stratified K-Fold và train–test")
fig.tight_layout()
save_figure(fig, CHART_COMPARE / "kfold_vs_train_test_scatter.png")

# 4) Boxplot F1-macro theo feature-selection tại k=4
fig, axes = plt.subplots(1, 2, figsize=(11.5, 5.2), sharey=True)
flow_specs = [
    (axes[0], kfold[kfold["k"] == 4], FS_ORDER, "Stratified K-Fold"),
    (axes[1], split[split["k"] == 4], [f for f in FS_ORDER if f != "raw"], "Train–test"),
]
palette = ["#A5A5A5", "#70AD47", "#FFC000", "#5B9BD5"]
for ax, data, methods, title in flow_specs:
    values = [data.loc[data["fs_method"] == method, "f1_macro"].to_numpy() for method in methods]
    boxes = ax.boxplot(
        values,
        tick_labels=[FS_DISPLAY[m] for m in methods],
        patch_artist=True,
        showmeans=True,
        meanprops={"marker": "D", "markerfacecolor": "white", "markeredgecolor": "black", "markersize": 4},
    )
    for patch, color in zip(boxes["boxes"], palette[: len(methods)]):
        patch.set_facecolor(color)
        patch.set_alpha(0.80)
    ax.set_title(title)
    ax.set_xlabel("Cấu hình đặc trưng")
    ax.grid(axis="y", alpha=0.20)
axes[0].set_ylabel("F1-macro")
fig.suptitle("Phân bố F1-macro theo phương pháp lựa chọn đặc trưng tại k=4")
fig.tight_layout()
save_figure(fig, CHART_COMPARE / "feature_selection_boxplot_k4.png")

# 5) Boxplot F1-macro theo mô hình tại k=4
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.2), sharey=True)
for ax, data, title in [
    (axes[0], kfold[kfold["k"] == 4], "Stratified K-Fold"),
    (axes[1], split[split["k"] == 4], "Train–test"),
]:
    values = [data.loc[data["model"] == model, "f1_macro"].to_numpy() for model in MODEL_ORDER]
    boxes = ax.boxplot(
        values,
        tick_labels=[MODEL_DISPLAY[m] for m in MODEL_ORDER],
        patch_artist=True,
        showmeans=True,
        meanprops={"marker": "D", "markerfacecolor": "white", "markeredgecolor": "black", "markersize": 4},
    )
    for index, patch in enumerate(boxes["boxes"]):
        patch.set_facecolor(plt.cm.Set3(index / len(MODEL_ORDER)))
        patch.set_alpha(0.85)
    ax.set_title(title)
    ax.set_xlabel("Mô hình")
    ax.tick_params(axis="x", rotation=25)
    ax.grid(axis="y", alpha=0.20)
axes[0].set_ylabel("F1-macro")
fig.suptitle("Phân bố F1-macro theo mô hình tại k=4")
fig.tight_layout()
save_figure(fig, CHART_COMPARE / "model_boxplot_k4.png")

CHART: outputs_LV\sec_03_classification_results\charts\kfold\best_f1_heatmap.png


CHART: outputs_LV\sec_03_classification_results\charts\train_test\best_f1_k4_k5_grouped_bar.png


CHART: outputs_LV\sec_03_classification_results\charts\comparison\kfold_vs_train_test_scatter.png


CHART: outputs_LV\sec_03_classification_results\charts\comparison\feature_selection_boxplot_k4.png


CHART: outputs_LV\sec_03_classification_results\charts\comparison\model_boxplot_k4.png


In [6]:
descriptions = {
    "tables/kfold/best_by_dataset_k.csv": "Cấu hình đại diện tốt nhất theo dataset và k trong K-Fold.",
    "tables/kfold/all_tied_best_configurations.csv": "Toàn bộ cấu hình đồng hạng F1 tốt nhất trong K-Fold.",
    "tables/train_test/best_k4_k5_wide.csv": "Bảng đối chiếu cấu hình tốt nhất giữa k=4 và k=5.",
    "tables/comparison/threshold_tradeoff.csv": "Đánh đổi hiệu năng, số mẫu và số lớp theo k.",
    "tables/comparison/feature_selection_k4.csv": "So sánh feature selection giữa hai luồng tại k=4.",
    "charts/kfold/best_f1_heatmap.png": "Heatmap best F1 theo dataset và k.",
    "charts/train_test/best_f1_k4_k5_grouped_bar.png": "Grouped bar best F1 tại k=4 và k=5.",
    "charts/comparison/kfold_vs_train_test_scatter.png": "Scatter đối chiếu K-Fold và train–test.",
    "charts/comparison/feature_selection_boxplot_k4.png": "Boxplot theo feature selection tại k=4.",
    "charts/comparison/model_boxplot_k4.png": "Boxplot theo mô hình tại k=4.",
}

artifact_rows = []
for path in sorted(EXPORT_ROOT.rglob("*")):
    if path.is_file() and path.name != "manifest.csv":
        relative = path.relative_to(EXPORT_ROOT).as_posix()
        artifact_rows.append({
            "artifact_group": relative.split("/")[0],
            "relative_path": relative,
            "suffix": path.suffix.lower(),
            "size_bytes": path.stat().st_size,
            "description": descriptions.get(relative, ""),
        })

manifest = pd.DataFrame(artifact_rows)
manifest.to_csv(EXPORT_ROOT / "manifest.csv", index=False, encoding="utf-8-sig")

expected_charts = [
    CHART_KFOLD / "best_f1_heatmap.png",
    CHART_SPLIT / "best_f1_k4_k5_grouped_bar.png",
    CHART_COMPARE / "kfold_vs_train_test_scatter.png",
    CHART_COMPARE / "feature_selection_boxplot_k4.png",
    CHART_COMPARE / "model_boxplot_k4.png",
]
for path in [*csv_exports.keys(), *expected_charts]:
    if not path.exists() or path.stat().st_size == 0:
        raise RuntimeError(f"Artifact chưa được sinh đúng: {path}")

print("\nHoàn tất:", len(manifest), "artifact")
print("K-Fold mean best F1 theo k:")
display(threshold_tradeoff[["k", "mean_best_f1", "mean_best_balanced_accuracy", "remaining_samples", "remaining_classes"]])
print("Train–test mean best F1:")
display(
    split_best_wide[["f1_macro_k4", "f1_macro_k5", "delta_f1_k5_minus_k4"]]
    .mean()
    .to_frame("mean")
)
display(manifest)


Hoàn tất: 96 artifact
K-Fold mean best F1 theo k:


,k,mean_best_f1,mean_best_balanced_accuracy,remaining_samples,remaining_classes
0,2,0.920499,0.926289,1586,68
1,4,0.930008,0.934577,1570,60
2,5,0.929868,0.934661,1558,57
3,6,0.934806,0.938365,1533,52


Train–test mean best F1:


,mean
f1_macro_k4,0.917727
f1_macro_k5,0.884410
delta_f1_k5_minus_k4,-0.033317


,artifact_group,relative_path,suffix,size_bytes,description
0,charts,charts/comparison/feature_selection_boxplot_k4...,.png,111940,Boxplot theo feature selection tại k=4.
1,charts,charts/comparison/kfold_vs_train_test_scatter.png,.png,249365,Scatter đối chiếu K-Fold và train–test.
2,charts,charts/comparison/model_boxplot_k4.png,.png,142525,Boxplot theo mô hình tại k=4.
3,charts,charts/kfold/best_f1_heatmap.png,.png,255573,Heatmap best F1 theo dataset và k.
4,charts,charts/kfold/by_k_feature_selection/k2/boruta_...,.png,627947,
...,...,...,...,...,...
91,tables,tables/train_test/dataset_model_line_charts/mo...,.csv,4948,
92,tables,tables/train_test/feature_selection_summary_al...,.csv,398,
93,tables,tables/train_test/feature_selection_summary_k4...,.csv,395,
94,tables,tables/train_test/model_summary_all_k.csv,.csv,743,


## Bi?u ?? theo dataset v? b?ng trung b?nh t??ng ?ng

M?i bi?u ?? c? ??nh m?t lu?ng ??nh gi?, ng??ng `k` v? ph??ng ph?p l?a ch?n ??c tr?ng. ???ng li?n bi?u di?n k?t qu? c?a t?ng m? h?nh tr?n c?c dataset; ???ng n?t ??t l? trung b?nh kh?ng tr?ng s? c?a ch?nh m? h?nh ?? tr?n c?c dataset hi?n c?. CSV ???c xu?t tr??c, sau ?? ???c ??c l?i ?? sinh bi?u ??.

In [7]:
import sys

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.sec03_line_charts import generate_protocol_line_artifacts

kfold_line_artifacts = generate_protocol_line_artifacts(
    data=kfold,
    protocol_key="kfold",
    protocol_title="Stratified K-Fold",
    table_root=TABLE_KFOLD,
    chart_root=CHART_KFOLD,
    dataset_order=DATASET_ORDER,
    dataset_display=DATASET_DISPLAY,
    fs_order=FS_ORDER,
    fs_display=FS_DISPLAY,
    model_order=MODEL_ORDER,
    model_display=MODEL_DISPLAY,
)

split_line_artifacts = generate_protocol_line_artifacts(
    data=split,
    protocol_key="train_test",
    protocol_title="Train--test",
    table_root=TABLE_SPLIT,
    chart_root=CHART_SPLIT,
    dataset_order=DATASET_ORDER,
    dataset_display=DATASET_DISPLAY,
    fs_order=FS_ORDER,
    fs_display=FS_DISPLAY,
    model_order=MODEL_ORDER,
    model_display=MODEL_DISPLAY,
)

# Rebuild manifest after exporting all CSV tables and charts.
artifact_rows = []
for path in sorted(EXPORT_ROOT.rglob("*")):
    if path.is_file() and path.name != "manifest.csv":
        relative = path.relative_to(EXPORT_ROOT).as_posix()
        if "dataset_model_line_charts" in relative:
            description = "Dữ liệu Dataset--Model và trung bình dùng cho biểu đồ đường."
        elif "by_k_feature_selection" in relative:
            description = "Biểu đồ mô hình theo dataset, có đường trung bình, cố định k và feature selection."
        elif "overview_by_k" in relative:
            description = "Tổng quan Accuracy và F1-macro theo feature selection tại một ngưỡng k."
        else:
            description = descriptions.get(relative, "")
        artifact_rows.append({
            "artifact_group": relative.split("/")[0],
            "relative_path": relative,
            "suffix": path.suffix.lower(),
            "size_bytes": path.stat().st_size,
            "description": description,
        })

manifest = pd.DataFrame(artifact_rows)
manifest.to_csv(EXPORT_ROOT / "manifest.csv", index=False, encoding="utf-8-sig")

expected_individual = 32 + 12  # K-Fold: 4 k × 4 FS × 2 metric; train--test: 2 × 3 × 2
actual_individual = (
    len(kfold_line_artifacts["individual_charts"])
    + len(split_line_artifacts["individual_charts"])
)
if actual_individual != expected_individual:
    raise RuntimeError(f"Thiếu biểu đồ đường: {actual_individual}/{expected_individual}")

print(f"Đã sinh {actual_individual} biểu đồ chi tiết và 6 biểu đồ tổng quan theo k.")
print(f"Manifest sau cập nhật: {len(manifest)} artifact.")
display(pd.read_csv(kfold_line_artifacts["mean_csv"]).head(14))
display(pd.read_csv(split_line_artifacts["mean_csv"]).head(14))

Đã sinh 44 biểu đồ chi tiết và 6 biểu đồ tổng quan theo k.
Manifest sau cập nhật: 96 artifact.


,k,fs_method,model,mean_accuracy_across_datasets,mean_balanced_accuracy_across_datasets,mean_f1_macro_across_datasets,mean_precision_across_datasets,mean_recall_across_datasets,n_datasets_in_mean
0,2,raw,nb,0.726876,0.602798,0.584676,0.601608,0.602798,12
1,2,raw,knn,0.706227,0.626725,0.608285,0.643862,0.626725,12
2,2,raw,svm,0.738174,0.631840,0.628261,0.672583,0.631840,12
3,2,raw,dt,0.650052,0.592512,0.578887,0.597663,0.592512,12
4,2,raw,rf,0.803561,0.745186,0.725583,0.731963,0.745186,12
5,2,raw,xgboost,0.791547,0.691748,0.679010,0.690887,0.691748,12
6,2,raw,ann,0.785519,0.732873,0.713424,0.722486,0.732873,11
7,2,mrmr_k50,nb,0.786144,0.728605,0.705971,0.712157,0.728605,12
8,2,mrmr_k50,knn,0.836498,0.771896,0.758785,0.767960,0.771896,12
9,2,mrmr_k50,svm,0.822387,0.785579,0.766801,0.771407,0.785579,12


,k,fs_method,model,mean_accuracy_across_datasets,mean_balanced_accuracy_across_datasets,mean_f1_macro_across_datasets,mean_precision_across_datasets,mean_recall_across_datasets,n_datasets_in_mean
0,4,mrmr_k50,nb,0.738208,0.711113,0.676803,0.669405,0.711113,12
1,4,mrmr_k50,knn,0.736983,0.686619,0.662683,0.664903,0.686619,12
2,4,mrmr_k50,svm,0.691305,0.659480,0.639141,0.644715,0.659480,12
3,4,mrmr_k50,dt,0.784614,0.802403,0.777442,0.791678,0.802403,12
4,4,mrmr_k50,rf,0.853522,0.843053,0.825450,0.828526,0.843053,12
5,4,mrmr_k50,xgboost,0.851445,0.828274,0.809183,0.812721,0.828274,12
6,4,mrmr_k50,ann,0.787501,0.778229,0.752448,0.757101,0.778229,12
7,4,mrmr_k75,nb,0.732963,0.701082,0.671115,0.660308,0.701082,12
8,4,mrmr_k75,knn,0.760271,0.704220,0.675773,0.681988,0.704220,12
9,4,mrmr_k75,svm,0.736476,0.720500,0.690353,0.697547,0.720500,12
